## Root Cause Analysis - Exp1

* Goals
  * Root Cause Analysis on individual RAG results --> 2 RAGs results comparison
  * Provide actionable suggestions

### Get Auto Eval Output

In [1]:
%load_ext autoreload
%autoreload 2

import os
import json
import psycopg2
import pandas as pd

from utils_root_cause import *

import warnings
warnings.filterwarnings('ignore')

# check saved output
DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")
conn = psycopg2.connect(DATABASE_URL_PUBLIC)
conn.autocommit = True
db_name = conn.get_dsn_parameters()['dbname']
print(f"Connected to database: {db_name}")
cur = conn.cursor()

Connected to database: railway


In [2]:
cur.execute("""
    SELECT count(distinct config_hash) from existing_rag_output
""")
rows = cur.fetchall()
print(rows[0][0])
print()

cur.execute("""
    SELECT count(distinct config_hash) from existing_auto_eval_output
""")
rows = cur.fetchall()
print(rows[0][0])

32

25


In [3]:
cur.execute("""
    SELECT
        t1.config_hash,
        t1.dataset,
        t1.embedding_model,
        t1.top_n_retrieval,
        t1.semantic_weight,
        t1.answer_gen_llm,
        rq.value->>'query' AS query,
        rq.value->>'context' AS context,
        rq.value->>'retrieved_content' AS retrieved_content,
        rq.value->>'same_context' AS same_context,
        rq.value->>'retrieval_quality_score' AS retrieval_quality_score,
        rq.value->>'rq_reasoning' AS rq_reasoning,
        aq.value->>'referenced_answer' AS referenced_answer,
        aq.value->>'ai_answer' AS ai_answer,
        aq.value->>'answer_quality_score' AS answer_quality_score,
        aq.value->>'aq_reasoning' AS aq_reasoning
    FROM existing_rag_output AS t1
    JOIN existing_auto_eval_output AS t2
        ON t1.config_hash = t2.config_hash,
    jsonb_array_elements(t2.retrieval_quality) WITH ORDINALITY AS rq(value, idx),
    jsonb_array_elements(t2.answer_quality) WITH ORDINALITY AS aq(value, idx)
    WHERE rq.idx = aq.idx
""")
rows = cur.fetchall()
col_names = [desc[0] for desc in cur.description]
df = pd.DataFrame(rows, columns=col_names)
print(df.shape)
df.head()

(750, 16)


,config_hash,dataset,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm,query,context,retrieved_content,same_context,retrieval_quality_score,rq_reasoning,referenced_answer,ai_answer,answer_quality_score,aq_reasoning
0,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,true,3,The retrieved content is essentially identical...,Have the check reissued to the proper payee.Ju...,To deposit a cheque issued to an associate in ...,2,The AI’s answer is relevant to the user’s ques...
1,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,true,3,The retrieved content fully matches the contex...,Sure you can. You can fill in whatever you wa...,"Yes, you can fill in your business name and ad...",2,The AI’s answer confirms that a business can u...
2,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,Sure you can. You can fill in whatever you wa...,false,2,The retrieved content acknowledges that a DBA ...,You're confusing a lot of things here. Company...,You can have one Employer Identification Numbe...,-1,The AI’s answer directly addresses the user’s ...
3,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,Set up a meeting with the bank that handles yo...,true,3,The retrieved content is identical to the cont...,"""I'm afraid the great myth of limited liabilit...",Establishing a relationship with a banker is c...,2,The AI answer addresses the user’s request and...
4,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,The time horizon for your 401K/IRA is essentia...,true,3,The retrieved content is essentially identical...,You should probably consult an attorney. Howev...,If your employer's 401(k) plan is terminated d...,2,The AI’s answer acknowledges that a 401(k) pla...


In [7]:
google_drive_folder_id = '1DLyuFLm0N6BwR2NvDoCEPPMHTzUK5xuv'
output_filename = 'rag_eval_output_v1.xlsx'
upload_to_google_drive(df, google_drive_folder_id, output_filename)

Overwritten file ID: 1XMeCrpFXT3kVIsR4d8VIfSFMZc5mJ-yi


In [4]:
cur.close()
conn.close()

In [6]:
pd.DataFrame(df[['retrieval_quality_score', 'answer_quality_score']].value_counts())

count
retrieval_quality_score answer_quality_score       
3                       2                       316
                        3                        97
0                       2                        62
3                       1                        54
                        4                        52
2                       -1                       24
                        2                        21
1                       2                        17
3                       -1                       17
                        0                        15
2                       1                        14
                        4                        12
0                       1                        12
                        4                        10
1                       -1                        6
0                       0                         4
1                       1                         4
                        4                         3
0                       3                         3
                        -1                        2
1                       3                         2
2                       0                         2
                        3                         1